In [ ]:
import cogsworth
import astropy.units as u
import astropy.constants as const
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import gala.dynamics as gd
import gala.potential as gp
from gala.units import galactic
from scipy.integrate import cumulative_trapezoid
from astropy.coordinates import SkyCoord
from scipy.ndimage import gaussian_filter
import seaborn as sns
import os

import sys
sys.path.append('../src')
import plotting, helpers
from importlib import reload

%config InlineBackend.figure_format = 'retina'


pd.options.display.max_columns = 999

plt.rc('font', family='serif')
plt.rcParams['text.usetex'] = False
fs = 24

# update various fontsizes to match
params = {'figure.figsize': (12, 8),
          'legend.fontsize': 0.7*fs,
          'legend.title_fontsize': 0.8*fs,
          'axes.labelsize': fs,
          'xtick.labelsize': 0.9 * fs,
          'ytick.labelsize': 0.9 * fs,
          'axes.linewidth': 1.1,
          'xtick.major.size': 7,
          'xtick.minor.size': 4,
          'ytick.major.size': 7,
          'ytick.minor.size': 4}
plt.rcParams.update(params)

In [ ]:
files = ["fiducial"]
labels = ["Fiducial"]
colours = ["tab:blue"]

In [ ]:
pops, data = helpers.load_postprocessed_pops(files, labels, colours)

Loading post-processed data for Fiducial from fiducial


In [ ]:
bound_pos = data["Fiducial"]["pos"]["BH"][~data["Fiducial"]["escaped"]["BH"]]

In [ ]:
(np.linalg.norm(bound_pos, axis=1) < 2.5).sum() / len(bound_pos)

np.float64(0.08442831204326877)

In [ ]:
min_dist = 100

while min_dist > 0.05:
    random_angle = np.random.uniform(0, 2 * np.pi)
    sun_loc = [8.2 * np.cos(random_angle), 8.2 * np.sin(random_angle), 0]
    min_dist = np.linalg.norm(data["Fiducial"]["pos"]["BH"] - sun_loc, axis=1).min()

In [ ]:
sun_loc = [8.2, 0, 0]
min_dist = np.linalg.norm(data["Fiducial"]["pos"]["BH"] - sun_loc, axis=1).min()
np.sort(np.linalg.norm(data["Fiducial"]["pos"]["BH"] - sun_loc, axis=1))[:10] * 1000

array([ 38.74938426,  82.78551409,  95.65136994,  97.29473449,
        98.08962235, 117.18076917, 119.81093145, 121.92428009,
       125.69112201, 131.83150877])

In [ ]:
nearest_ind = np.linalg.norm(data["Fiducial"]["pos"]["BH"] - sun_loc, axis=1).argmin()
nearest_pos = data["Fiducial"]["pos"]["BH"][nearest_ind]
nearest_vel = data["Fiducial"]["vel"]["BH"][nearest_ind]
nearest_mass = data["Fiducial"]["mass"]["BH"][nearest_ind]

In [ ]:
nearest_mass

np.float64(16.962627324868507)

In [ ]:
nearest_pos[0]

np.float64(8.219184960387464)

In [ ]:
from astropy.coordinates import get_sun
from astropy.time import Time

get_sun(Time.now()).galactocentric.represent_as("cartesian")

<CartesianRepresentation (x, y, z) in AU
    (-1.67527726e+09, -0.00048728, 4290307.97208976)>

In [ ]:
SkyCoord(x=-nearest_pos[0]*u.kpc, y=nearest_pos[1]*u.kpc, z=nearest_pos[2]*u.kpc,
         v_x=nearest_vel[0]*u.km/u.s, v_y=nearest_vel[1]*u.km/u.s, v_z=nearest_vel[2]*u.km/u.s,
         representation_type="cartesian", frame="galactocentric").icrs

<SkyCoord (ICRS): (ra, dec, distance) in (deg, deg, kpc)
    (81.83919083, 40.75380025, 0.09954551)
 (pm_ra_cosdec, pm_dec, radial_velocity) in (mas / yr, mas / yr, km / s)
    (-55.1822208, -165.12352812, 54.61760402)>

In [ ]:
data["Fiducial"]["vel"]["BH"][np.linalg.norm(data["Fiducial"]["pos"]["BH"] - sun_loc, axis=1).argmin()]

array([-54.60540333, 208.37622818, -53.78454848])

In [ ]:
min_dist * 1000

np.float64(38.74938426416195)

In [ ]:
companions = data["Fiducial"]["companion"]["BH"][(data["Fiducial"]["sep"]["BH"] > 0) & (data["Fiducial"]["primary"]["BH"])]
uni, count = np.unique(companions, return_counts=True)
uni, count

print("Number of BH + star:", count[uni < 10].sum())
print("Number of BH + WD:", count[(uni >= 10) & (uni <= 12)].sum())
print("Number of BH + NS:", count[uni == 13].sum())
print("Number of BH + BH:", count[uni == 14].sum())

Number of BH + star: 65
Number of BH + WD: 3773
Number of BH + NS: 1505
Number of BH + BH: 41724


In [ ]:
companions = data["Fiducial"]["companion"]["NS"][(data["Fiducial"]["sep"]["NS"] > 0) & (data["Fiducial"]["primary"]["NS"])]
uni, count = np.unique(companions, return_counts=True)

print("Number of NS + star:", count[uni < 10].sum())
print("Number of NS + WD:", count[(uni >= 10) & (uni <= 12)].sum())
print("Number of NS + NS:", count[uni == 13].sum())
print("Number of NS + BH:", count[uni == 14].sum())

Number of NS + star: 592
Number of NS + WD: 4218
Number of NS + NS: 230
Number of NS + BH: 3
